# 03 · Telemetri & kualitas data

Menilai kesehatan perangkat dari aliran telemetri: **kelengkapan data**, **celah (offline)**, dan perilaku sensor.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.io as pio

from dashboard import charts, data, metrics

pio.renderers.default = "notebook_connected"
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
tel = data.load_telemetry(max_rows=300_000)
print(f'{len(tel):,} bacaan, {tel.created_at.min()} s/d {tel.created_at.max()}')

## Kelengkapan data per jam

Dibandingkan dengan jumlah ideal `3600 / interval` (interval dari `TELEMETRY_INTERVAL_SECONDS`).

In [ ]:
cov = metrics.telemetry_coverage(tel)
charts.fig_coverage(cov)

In [ ]:
cov.groupby('unit_label')['kelengkapan_persen'].mean().sort_values()

## Celah data (perangkat diduga offline)

In [ ]:
g = metrics.gaps(tel)
g.head(10)

## Komposisi state deteksi

In [ ]:
charts.fig_state_mix(metrics.state_mix(tel))

In [ ]:
charts.fig_confirmed_share(metrics.telemetry_hourly(tel))

## Jarak sensor satu unit (rata-rata per menit)

In [ ]:
unit = sorted(tel['unit_label'].unique())[0]
per_menit = (tel[tel['unit_label'] == unit].set_index('created_at')[['distance_cm']]
             .resample('1min').mean().dropna().reset_index().assign(unit_label=unit))
charts.fig_distance(per_menit, tolerant_cm=10)

## Hubungan telemetri dan alert

Perbandingan sederhana jumlah bacaan `confirmed` per unit dengan jumlah alert yang tercatat.

In [ ]:
alerts = data.load_alerts()
conf = tel[tel.detection_state == 'confirmed'].groupby('unit_label').size().rename('bacaan_confirmed')
al = alerts.groupby('unit_label').size().rename('alert')
pd.concat([conf, al], axis=1).fillna(0).astype(int)